# Modélisation et MLflow — Partie 4

Ce notebook affiche les artefacts déjà produits. Il ne relance pas l'entraînement.

Commande des scores : `python -m src.models.train --mode full --n_rows 100000`
(fichier `reports/metrics.json`, date `2026-10-09T13:14:36Z`).


In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px

ROOT = Path("..").resolve()
REPORTS = ROOT / "reports"


def style_plotly(fig):
    fig.update_layout(title_font_size=16, template="plotly_dark")
    return fig


## Scores de validation


In [2]:
metrics = json.loads((REPORTS / "metrics.json").read_text(encoding="utf-8"))
print(metrics["date"], metrics["period_min"], "->", metrics["period_max"])
print("n_loaded", metrics["n_rows_loaded"], "n_train", metrics["n_train"], "n_val", metrics["n_val"])
scores = pd.DataFrame(metrics["models"])[
    ["name", "rmse", "mae", "r2", "rmse_card", "mae_card", "r2_card"]
].sort_values("rmse")
display(scores)


2026-10-09T13:14:36Z 2023-01-01 00:00:00 -> 2023-01-02 13:35:06
n_loaded 100000 n_train 72735 n_val 18751


,name,rmse,mae,r2,rmse_card,mae_card,r2_card
2,grid_001,0.085689,0.052029,0.640960,0.100172,0.070527,0.175483
3,grid_005,0.085713,0.051599,0.640759,0.100212,0.070219,0.174826
7,embedding_lightgbm,0.085746,0.053449,0.640486,0.100168,0.071328,0.175549
4,grid_004,0.085776,0.052537,0.640232,0.100236,0.070494,0.174417
5,grid_003,0.085812,0.052180,0.639928,0.100314,0.070249,0.173133
6,grid_002,0.086272,0.055066,0.636060,0.100648,0.071304,0.167626
1,baseline_linear,0.097639,0.069525,0.533839,0.105965,0.077372,0.077365
0,baseline_mean,0.143131,0.123788,-0.001740,0.125168,0.101928,-0.287345


## Importance (gain LightGBM)


In [3]:
importance = pd.read_csv(REPORTS / "feature_importance.csv").sort_values("importance")
display(importance)
fig = px.bar(
    importance,
    x="importance",
    y="feature",
    orientation="h",
    title="Importance des variables",
)
style_plotly(fig)
fig.show()


,feature,importance
12,pickup_quarter,0.000000
13,is_rush_hour,0.000000
14,is_late_night,0.000000
15,pickup_weekday_sin,0.000000
11,pickup_month,0.000000
16,pickup_weekday_cos,0.000000
10,is_weekend,0.122634
9,passenger_count,1.769239
8,pickup_weekday,1.796122
7,pickup_hour_sin,4.458902


C:\Users\kevin\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## Runs exportés

`reports/mlflow_runs.csv` reprend la même commande. Les courbes d'apprentissage
et les résidus ne sont pas dans cet export : ce notebook ne les invente pas.


In [4]:
runs = pd.read_csv(REPORTS / "mlflow_runs.csv").sort_values("rmse")
display(runs[["name", "rmse", "mae", "r2", "rmse_card"]])


,name,rmse,mae,r2,rmse_card
0,grid_001,0.085689,0.052029,0.640960,0.100172
1,grid_005,0.085713,0.051599,0.640759,0.100212
2,embedding_lightgbm,0.085746,0.053449,0.640486,0.100168
3,grid_004,0.085776,0.052537,0.640232,0.100236
4,grid_003,0.085812,0.052180,0.639928,0.100314
5,grid_002,0.086272,0.055066,0.636060,0.100648
6,baseline_linear,0.097639,0.069525,0.533839,0.105965
7,baseline_mean,0.143131,0.123788,-0.001740,0.125168


## Registre


In [5]:
registry = json.loads((REPORTS / "registry.json").read_text(encoding="utf-8"))
display(pd.Series(registry))


model_name                           TipPredictor
version                                         1
stage                                  Production
model_uri         models:/TipPredictor/Production
source_run_id    09ac8b9ff161415bb8f1042ec8287421
run_name                                 grid_001
rmse_val                                 0.085689
n_train                                     72735
dtype: object